In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, r2_score

In [ ]:
df = pd.read_csv("output/w2_miti_global_scores_20251222_v003.csv")


In [ ]:
df_wide = (
    df
    .pivot(index="session_id", columns="miti_dimension", values="score")
    .reset_index()
)

df_wide["user_id_raw"] =  df_wide["session_id"].str.replace("MI-MAINEXP-", "", regex=False).astype(int)


In [ ]:
df_wide.head()

In [ ]:
survey_data = pd.read_stata("/path/to/project/data/processed/main_social_media/clean_data.dta")

In [ ]:
survey_data_merge = survey_data[["T", "user_id_raw"]]
survey_data_merge["user_id_raw"] = survey_data_merge["user_id_raw"].astype(int)

In [ ]:
df_clean = df_wide.merge(survey_data_merge, on="user_id_raw", how="left", validate="one_to_one")

In [ ]:
df_clean.head()
df_clean = df

In [ ]:
df_clean.describe()

In [ ]:
score_cols = [
    "Cultivating Change Talk",
    "Empathy",
    "Partnership",
    "Softening Sustain Talk",
]

# Row count before
n_before = len(df_clean)

# Build condition: all scores between 0 and 5
mask = df_clean[score_cols].apply(lambda s: s.between(0, 5)).all(axis=1)

# Filter
df_filter = df_clean.loc[mask].copy()

# Row count after
n_after = len(df_filter)

print(f"Removed {n_before - n_after} rows out of {n_before}")


df_filter["T"] = df_filter["T"].cat.remove_unused_categories()


# Sanity check


In [ ]:
df_filter["T"].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

score_cols = [
    "Cultivating Change Talk",
    "Empathy",
    "Partnership",
    "Softening Sustain Talk",
]

plt.style.use("default")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    sns.kdeplot(
        data=df_filter,
        x=col,
        hue="T",
        common_norm=False,
        fill=False,
        ax=axes[i]
    )
    axes[i].set_title(f"Density of {col} by Treatment")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Density")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    sns.histplot(
        data=df_filter,
        x=col,
        hue="T",
        discrete=True,
        stat="count",          # <-- COUNTS
        multiple="dodge",      # side-by-side bars
        shrink=0.8,
        ax=axes[i]
    )
    axes[i].set_title(f"Counts of {col} by Treatment")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.show()